# 🧚 동화 AI 서버 — Colab T4

**순서대로 실행하세요 (셀 1 → 5)**

> ⚠️ 런타임 유형: 수정 → 런타임 유형 변경 → **T4 GPU** 선택

In [ ]:
# 셀 1: 패키지 설치
!pip install -q fastapi uvicorn peft transformers accelerate bitsandbytes nest_asyncio
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb -q
import torch
print('✅ 설치 완료')
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '없음')

In [ ]:
# 셀 2: Drive 마운트 + final_model 복사
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

# ✏️ 필요시 경로 수정
DRIVE_MODEL = '/content/drive/MyDrive/fairytale_ai/final_model'
LOCAL_MODEL  = '/content/final_model'

if not os.path.exists(DRIVE_MODEL):
    print('❌ 경로를 찾을 수 없어요. 아래 목록에서 실제 경로 확인:')
    !ls /content/drive/MyDrive/
else:
    if os.path.exists(LOCAL_MODEL):
        shutil.rmtree(LOCAL_MODEL)
    shutil.copytree(DRIVE_MODEL, LOCAL_MODEL)
    print('✅ 복사 완료:', os.listdir(LOCAL_MODEL))

In [ ]:
%%writefile /content/server.py
import os, json, time
from pathlib import Path
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel

app = FastAPI(title='동화 AI API', version='1.0.0')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

BASE_MODEL   = 'Qwen/Qwen2.5-3B-Instruct'
ADAPTER_PATH = Path('/content/final_model')

cfg = ADAPTER_PATH / 'adapter_config.json'
if cfg.exists():
    BASE_MODEL = json.loads(cfg.read_text()).get('base_model_name_or_path', BASE_MODEL)

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype  = torch.float16 if device == 'cuda' else torch.float32
print(f'[서버] 디바이스: {device}  모델: {BASE_MODEL}')

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.float16
) if device == 'cuda' else None

print('[서버] 모델 로딩 중...')
tokenizer = AutoTokenizer.from_pretrained(str(ADAPTER_PATH), trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb, device_map=device,
    torch_dtype=dtype, trust_remote_code=True, low_cpu_mem_usage=True
)
print('[서버] LoRA 어댑터 적용 중...')
model = PeftModel.from_pretrained(base_model, str(ADAPTER_PATH))
model.eval()
print('[서버] 준비 완료 ✓')

def call_model(prompt, max_tokens=800):
    text = tokenizer.apply_chat_template(
        [{'role': 'user', 'content': prompt}],
        tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_tokens,
            temperature=0.85, top_p=0.92,
            repetition_penalty=1.15, do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

SYS = '당신은 어린이를 위한 한국어 동화 작가 AI입니다. 쉽고 따뜻한 문체, 짧은 문장, 폭력/공포 금지, 긍정적 메시지.'

def p_story(genre, age, prompt, prev=''):
    ctx = f'이전 줄거리: {prev}\n' if prev else ''
    return f'{SYS}\n장르:{genre} 연령:{age}\n{ctx}요청:{prompt}\n\n2-3 문단으로 동화를 작성하세요.'

def p_choices(story, genre, age):
    s = story[-500:]
    return f'{SYS}\n[이야기]\n{s}\n\n선택지 3개를 아래 JSON 형식으로만 답하세요:\n' + '{"choices":["선택지1","선택지2","선택지3"]}'

def p_cont(story, choice, genre, age):
    s = story[-600:]
    return f'{SYS}\n장르:{genre} 연령:{age}\n[앞이야기]\n{s}\n[선택]{choice}\n\n2-3 문단으로 이어쓰세요.'

def p_vocab(text, age):
    t = text[:600]
    return f'{age} 아이를 위한 단어 3-5개를 JSON으로만:\n' + '{"words":[{"hard":"단어","easy":"쉬운표현","definition":"뜻"}]}' + f'\n[동화]\n{t}'

def p_psych(choices):
    cs = '\n'.join(f'- {c}' for c in choices)
    return f'아이 선택:\n{cs}\n\n성격 분석 JSON으로만:\n' + '{"type":"탐험가","description":"...","traits":{"모험적":85,"친절함":70,"용감함":90,"창의적":65,"협동심":60}}'

class StoryReq(BaseModel): genre:str; age:str; prompt:str
class ContReq(BaseModel):  story_id:str; story_so_far:str; choice:str; genre:str; age:str
class PsychReq(BaseModel): story_id:str; choices_made:list

@app.get('/health')
def health(): return {'status': 'ok', 'device': device}

@app.post('/story/start')
async def start(req: StoryReq):
    try:
        text = call_model(p_story(req.genre, req.age, req.prompt), 600)
        try:    choices = json.loads(call_model(p_choices(text, req.genre, req.age), 200)).get('choices', [])
        except: choices = ['계속 나아간다', '도움을 요청한다', '생각해본다']
        try:    vocab = json.loads(call_model(p_vocab(text, req.age), 300)).get('words', [])
        except: vocab = []
        return {'story_id': f'story_{int(time.time())}', 'story_text': text,
                'choices': choices, 'vocab': vocab, 'chapter': 1}
    except Exception as e: raise HTTPException(500, str(e))

@app.post('/story/continue')
async def cont(req: ContReq):
    try:
        new = call_model(p_cont(req.story_so_far, req.choice, req.genre, req.age), 600)
        upd = req.story_so_far + '\n\n' + new
        try:    choices = json.loads(call_model(p_choices(upd, req.genre, req.age), 200)).get('choices', [])
        except: choices = ['계속 나아간다', '도움을 요청한다', '다른 방법']
        try:    vocab = json.loads(call_model(p_vocab(new, req.age), 300)).get('words', [])
        except: vocab = []
        return {'story_id': req.story_id, 'new_text': new,
                'choices': choices, 'vocab': vocab, 'choice_made': req.choice}
    except Exception as e: raise HTTPException(500, str(e))

@app.post('/story/psych')
async def psych(req: PsychReq):
    try:
        if len(req.choices_made) < 2:
            return {'type':'탐험가','description':'더 많은 선택이 필요해요!',
                    'traits':{'모험적':50,'친절함':50,'용감함':50,'창의적':50,'협동심':50}}
        try:    return json.loads(call_model(p_psych(req.choices_made), 300))
        except: return {'type':'용감한 탐험가','description':'멋진 성격이에요!',
                        'traits':{'모험적':80,'친절함':70,'용감함':85,'창의적':65,'협동심':60}}
    except Exception as e: raise HTTPException(500, str(e))

In [ ]:
# 셀 4: 서버 실행 + cloudflared 터널
import nest_asyncio, uvicorn, threading, sys, time, subprocess, re, os
nest_asyncio.apply()

# ── 서버 백그라운드 실행
def run_server():
    sys.path.insert(0, '/content')
    uvicorn.run('server:app', host='0.0.0.0', port=8000, log_level='warning')

threading.Thread(target=run_server, daemon=True).start()
print('서버 시작 중... (모델 로딩 포함 1~2분)')
time.sleep(8)

# ── cloudflared: 로그파일 방식
log_file = '/tmp/cf.log'
if os.path.exists(log_file):
    os.remove(log_file)

subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8000',
     '--logfile', log_file, '--loglevel', 'info'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)

# URL 나올 때까지 최대 40초 대기
SERVER_URL = None
for i in range(40):
    time.sleep(1)
    if os.path.exists(log_file):
        log = open(log_file).read()
        match = re.search(r'https://[^\s]+\.trycloudflare\.com', log)
        if match:
            SERVER_URL = match.group()
            break
    if i % 5 == 4:
        print(f'{i+1}초 대기 중...')

if SERVER_URL:
    print()
    print('=' * 55)
    print(f'  🚀 서버 URL: {SERVER_URL}')
    print('=' * 55)
    print()
    print('▶ Flutter api_service.dart 에 붙여넣기:')
    print(f"  static const String baseUrl = '{SERVER_URL}';")
    print()
    print('⚠️  이 탭을 닫으면 서버가 꺼져요!')
else:
    print('❌ URL 발급 실패. cloudflared 로그:')
    if os.path.exists(log_file):
        print(open(log_file).read()[-1000:])

In [ ]:
# 셀 5: 동화 생성 테스트
import requests

r = requests.get(f'{SERVER_URL}/health', timeout=10)
print('서버 상태:', r.json())

print('\n동화 생성 테스트 중... (T4 기준 10~20초)')
r2 = requests.post(f'{SERVER_URL}/story/start', json={
    'genre': '판타지', 'age': '초등_저학년', 'prompt': '용을 만난 아이'
}, timeout=120)

if r2.status_code == 200:
    data = r2.json()
    print('\n✅ 성공!')
    print('\n[동화]')
    print(data['story_text'][:400])
    print('\n[선택지]')
    for c in data.get('choices', []):
        print(f'  👉 {c}')
else:
    print('❌ 오류:', r2.text)